In [10]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from scipy.stats import fisher_exact
import itertools
from statsmodels.stats.multitest import multipletests

df = pd.read_excel('caffe_with_coffee.xlsx')

In [11]:
transactions = (
    df.groupby('Bill Number')["Item Desc"]
        .apply(list)
        .tolist()
)

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_items = pd.DataFrame(te_ary, columns=te.columns_)

min_count = 10
common_items = df_items.columns[df_items.sum() >= min_count]
df_items = df_items[common_items]

print(f"Kept {len(common_items)} common items out of {len(te.columns_)} total.")



Kept 389 common items out of 562 total.


In [ ]:
def fisher_pair(df, itemA, itemB):
    a = ((df[itemA]) & (df[itemB])).sum()
    b = ((df[itemA]) & (~df[itemB])).sum()
    c = ((~df[itemA]) & (df[itemB])).sum()
    d = ((~df[itemA]) & (~df[itemB])).sum()
    oddsratio, p = fisher_exact([[a, b], [c, d]], alternative='greater')
    return {'itemA': itemA, 'itemB': itemB, 'a': a, 'oddsratio': oddsratio, 'p_value': p}


results = []
items = df_items.columns.tolist()

for itemA, itemB in itertools.combinations(items, 2):
    results.append(fisher_pair(df_items, itemA, itemB))

res_df = pd.DataFrame(results)
res_df['p_adj'] = multipletests(res_df['p_value'], method='fdr_bh')[1]
res_df = res_df.sort_values('p_adj')


In [ ]:
res_df['p_adj'] = multipletests(res_df['p_value'], method='fdr_bh')[1]

res_df = res_df.sort_values('p_adj')
significant = res_df[(res_df['p_adj'] < 0.05) & (res_df['oddsratio'] > 1)]

print("\nTop significant item pairs:")
print(significant[['itemA', 'itemB', 'a', 'oddsratio', 'p_adj']].head(20))


Top significant item pairs:
                         itemA                           itemB    a  \
49409       HERB ROAST CHICKEN  LEMON INFUSED CHAR GRILLED VEG   73   
17944              CAFFE LATTE                HAZELNUT FLAVOUR  113   
70530                 RED BULL                         SAMBUCA  343   
16625          BUTTERED TOASTS                  KHEEMA GHOTALA   48   
49454       HERB ROAST CHICKEN                ORANGE ARRABIATA   51   
6197             B.M.T. PANINI                           FRIES  151   
41243                    FRIES    TRADITIONAL ITALIAN CRUSTINI   59   
18149              CAFFE LATTE                 VANILLA FLAVOUR   40   
18955               CAPPUCCINO                HAZELNUT FLAVOUR   96   
17840              CAFFE LATTE                 CARAMEL FLAVOUR   44   
25608                  CHICKEN                  SAIGON NOODLES   17   
10553              BERRY BLAST                COOL CALIFORNICA  125   
15614         BUN MASKA & CHAI             MASAL